# 03 Spell Check

Reads `processed/02_contractions.csv` and corrects spelling, one word at a time, in this order:

0. **Your hand fixes** in `output_data/spelling_hand_fixes.csv` (columns `original`, `correction`; a correction equal to the original means "keep as written") are applied first, along with contractions typed without an apostrophe ("dont" becomes "do not").
1. **Checked entries** in the hand-edited dictionary (`output_data/spelling_dict_edited.xlsx`, rows with `checked = yes`) are used as they are. The other entries in that file are only the old spell checker's first suggestion (it turned "dont" into "font" and "texas" into "taxes"), so they are not trusted.
2. **Known words** are left alone: a standard English word list (including American spellings and inflected forms), the SymSpell frequency list, and the cue words.
3. **Unknown words** (at least 4 letters) are corrected only when the fix is safe:
   - SymSpell finds a word one edit away that starts with the same letter, **and** the old spell checker suggested the same word, or
   - the word was used only once or twice (so it is probably a typo, not a name or slang), or
   - the old suggestion is just the word split into two real words of 4+ letters ("waferthin" becomes "wafer thin"), or
   - SymSpell has no suggestion and the old checker's is within two edits and starts with the same letter ("jewlery" becomes "jewelry").
4. Everything else is left as written and listed for review, with both suggestions.

Nothing is dropped. Output: `processed/03_spelling.csv`.

Review files:
- `processed/03_spelling_changes.csv`: every correction, how it was made, and how many times it was used.
- `processed/03_spelling_unresolved.csv`: unknown words that were not changed with the suggestions.

In [1]:
import pandas as pd
import re
import os
import symspellpy
from collections import Counter
from nltk.corpus import words as nltk_words, wordnet as wn
from symspellpy import SymSpell, Verbosity
from symspellpy.editdistance import EditDistance, DistanceAlgorithm

In [2]:
data = pd.read_csv("processed/02_contractions.csv", dtype=str, keep_default_na=False)
TOKEN = re.compile(r"[a-z]+(?:'[a-z]+)*")

counts = Counter()
for text in data["answer_contractions"]:
    counts.update(TOKEN.findall(text))
print(f"{len(data):,} answers, {sum(counts.values()):,} words, {len(counts):,} different words")

185,824 answers, 1,597,743 words, 45,118 different words


## Word lists

In [3]:
# 1. hand-edited dictionary
manual = pd.read_excel("output_data/spelling_dict_edited.xlsx", dtype=str)
print(manual["checked"].value_counts(dropna=False).to_dict())
ok = manual["spelling.errors"].notna() & manual["correction"].notna() & (manual["checked"] == "yes")
MANUAL = {str(e).lower(): str(c).lower() for e, c in zip(manual.loc[ok, "spelling.errors"], manual.loc[ok, "correction"])}
OLD_SUGGESTION = {
    str(e).lower(): str(c).lower()
    for e, c in zip(manual.loc[manual["checked"] != "yes", "spelling.errors"], manual.loc[manual["checked"] != "yes", "correction"])
    if pd.notna(e) and pd.notna(c)
}
print(f"{len(MANUAL):,} checked words in the hand-edited dictionary; the other {len(OLD_SUGGESTION):,} are only used to confirm SymSpell")

# 2. known words
freq_file = os.path.join(os.path.dirname(symspellpy.__file__), "frequency_dictionary_en_82_765.txt")
FREQ = {line.split()[0]: int(line.split()[1]) for line in open(freq_file)}
ENGLISH = {w.lower() for w in nltk_words.words()}
CUES = set(data["cue"])

def is_known(word):
    return word in FREQ or word in ENGLISH or word in CUES or wn.morphy(word) is not None

NO_APOSTROPHE = {
    "dont": "do not", "doesnt": "does not", "didnt": "did not", "isnt": "is not", "arent": "are not",
    "wasnt": "was not", "werent": "were not", "cant": "can not", "wont": "will not", "couldnt": "could not",
    "wouldnt": "would not", "shouldnt": "should not", "havent": "have not", "hasnt": "has not",
    "im": "i am", "youre": "you are", "theyre": "they are", "thats": "that is", "whats": "what is",
    "ive": "i have", "theres": "there is", "hes": "he is", "shes": "she is", "lets": "let us",
}
HAND_FIXES = {}
if os.path.exists("output_data/spelling_hand_fixes.csv"):
    fixes = pd.read_csv("output_data/spelling_hand_fixes.csv", dtype=str, keep_default_na=False)
    HAND_FIXES = dict(zip(fixes["original"].str.lower(), fixes["correction"].str.lower()))
print(f"{len(HAND_FIXES)} hand fixes")
edit_distance = EditDistance(DistanceAlgorithm.DAMERAU_OSA_FAST)

sym = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)
sym.load_dictionary(freq_file, term_index=0, count_index=1)

{'False': 12963, 'yes': 3032}
3,032 checked words in the hand-edited dictionary; the other 12,963 are only used to confirm SymSpell
135 hand fixes


True

## Decide a correction for every different word

In [4]:
SUGGESTIONS = {}

def decide(word):
    """Returns (correction, method). Correction equals the word if nothing changes."""
    if word in HAND_FIXES:
        return HAND_FIXES[word], "hand_fix"
    if word in NO_APOSTROPHE:
        return NO_APOSTROPHE[word], "no_apostrophe"
    if word in MANUAL:
        return MANUAL[word], "hand_dictionary"
    if "'" in word or len(word) < 2 or is_known(word):
        return word, "unchanged"
    if len(word) < 4:
        return word, "unresolved"
    old = OLD_SUGGESTION.get(word)
    best = sym.lookup(word, Verbosity.TOP, max_edit_distance=1, include_unknown=False)
    guess = best[0].term if best and best[0].term[0] == word[0] else None
    if guess and old == guess:
        return guess, "symspell_confirmed"
    if guess and counts[word] <= 2:
        return guess, "symspell_rare"
    if old and " " in old and old.replace(" ", "") == word and counts[word] <= 2:
        parts = old.split()
        if len(parts) == 2 and all(len(x) >= 4 and x in FREQ for x in parts):
            return old, "split_word"
    if old and " " not in old and not guess and old[0] == word[0] and edit_distance.compare(word, old, 2) != -1:
        return old, "old_suggestion_close"
    SUGGESTIONS[word] = (guess or "", old or "")
    return word, "unresolved"

decisions = {w: decide(w) for w in counts}
methods = Counter(m for _, m in decisions.values())
print(methods)

Counter({'unchanged': 33260, 'symspell_confirmed': 3857, 'hand_dictionary': 2497, 'symspell_rare': 2387, 'unresolved': 1854, 'old_suggestion_close': 806, 'split_word': 299, 'hand_fix': 135, 'no_apostrophe': 23})


In [5]:
changes = pd.DataFrame(
    [(w, c, m, counts[w]) for w, (c, m) in decisions.items() if c != w],
    columns=["original", "correction", "method", "count"],
).sort_values(["method", "count"], ascending=[True, False])
changes.to_csv("processed/03_spelling_changes.csv", index=False)

unresolved = pd.DataFrame(
    [(w, counts[w], *SUGGESTIONS.get(w, ("", ""))) for w, (c, m) in decisions.items() if m == "unresolved"],
    columns=["word", "count", "symspell_suggestion", "old_suggestion"],
).sort_values("count", ascending=False)
unresolved.to_csv("processed/03_spelling_unresolved.csv", index=False)

print(changes.groupby("method").agg(different_words=("original", "size"), times_used=("count", "sum")))
print(f"{len(unresolved):,} unresolved words ({unresolved['count'].sum():,} uses)")

                      different_words  times_used
method                                           
hand_dictionary                   931        1600
hand_fix                          135         736
no_apostrophe                      23         858
old_suggestion_close              806        1081
split_word                        299         340
symspell_confirmed               3857        6308
symspell_rare                    2387        2566
1,854 unresolved words (2,507 uses)


## Apply to every answer

In [6]:
def correct(text):
    return TOKEN.sub(lambda m: decisions[m.group(0)][0], text)

data["answer_spelled"] = data["answer_contractions"].map(correct)
n_changed = (data["answer_spelled"] != data["answer_contractions"]).sum()
print(f"{n_changed:,} of {len(data):,} answers changed")

11,924 of 185,824 answers changed


In [7]:
out = data[["row_id", "subject_id", "cue", "answer_contractions", "answer_spelled"]]
out.to_csv("processed/03_spelling.csv", index=False)
print(f"Saved {len(out):,} rows to processed/03_spelling.csv")
data.loc[data["answer_spelled"] != data["answer_contractions"], ["answer_contractions", "answer_spelled"]].sample(8, random_state=1)

Saved 185,824 rows to processed/03_spelling.csv


,answer_contractions,answer_spelled
185029,"numbers, soultion, letters symboles","numbers, solution, letters symbols"
159543,"self, huge, id, what perceive as your center, ...","self, huge, id, what perceive as your center, ..."
182043,"brains ability to bring back occurences, thing...","brains ability to bring back occurrences, thin..."
95376,"read, recipies, cook food","read, recipes, cook food"
12043,failure to do something example=didnt do your ...,failure to do something example=did not do you...
181794,"being controled by a single thought or idea, c...","being controled by a single thought or idea, c..."
137237,"shouldnt, do not, no","should not, do not, no"
90910,"100 years, age, 1/10th of a millenium","100 years, age, 1/10th of a millennium"
